# Model E — Patient-Facing Guidance

Translates system decisions into a calm, clear next-steps message for the patient.
No training data required — uses Gemini with few-shot examples and hard guardrails.

No diagnosis. No medical advice. Never overrides priority.

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=genai.types.GenerationConfig(temperature=0.3, max_output_tokens=180),
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"},
    ],
)

In [ ]:
@dataclass
class GuidanceInput:
    priority:             str            = "MEDIUM"
    chief_complaint:      str            = ""
    duration:             str            = ""
    severity:             str            = ""
    associated_symptoms:  List[str]      = field(default_factory=list)
    red_flags_present:    Optional[bool] = None
    patient_language:     str            = "english"
    location_context:     str            = ""
    uncertainty_flag:     bool           = False

In [ ]:
FALLBACK_TEMPLATES = {
    "HIGH":    "Your symptoms need prompt medical attention. Please proceed to the nearest staff member immediately. If you feel worse, notify a staff member right away.",
    "MEDIUM":  "Thank you for sharing your symptoms. A healthcare professional will see you shortly. Please take a seat and wait to be called. If your condition worsens, let a staff member know.",
    "LOW":     "Thank you. Your information has been recorded. Please wait and you will be called when it is your turn. If you have any concerns, speak to a staff member.",
    "UNKNOWN": "Thank you. Please wait while we complete the next step. A staff member will assist you shortly.",
}

FALLBACK_TEMPLATES_KIN = {
    "HIGH":    "Ibimenyetso byawe bisaba ubuvuzi bwihutirwa. Mwihangane, muzajya guturikirwa na muganga vuba. Niba imiterere yawe ihinduka, menyesha umukozi vuba.",
    "MEDIUM":  "Murakoze gushiraho ibimenyetso byanyu. Muganga azababona vuba. Mwicare mukomeze gutegereza. Niba hari impinduka, menyesha umukozi.",
    "LOW":     "Murakoze. Amakuru yanyu yanditswe. Mwicare mukomeze gutegereza kugeza igihe muzahamagalwa. Niba mufite ibibazo, mubwire umukozi.",
    "UNKNOWN": "Murakoze gutanga amakuru. Mwihangane turangije intambwe ikurikira. Umukozi azabafasha vuba.",
}

FORBIDDEN_PHRASES = [
    "you have", "you may have", "this is", "this could be", "diagnosis",
    "heart attack", "stroke", "cancer", "infection",
    "take", "medication", "rest", "drink water", "avoid",
    "life-threatening", "critical",
]

In [ ]:
SYSTEM_PROMPT = """\
You are a healthcare system assistant in a hospital.
Explain the next steps to a patient using only the system's decision.
Rules:
- No disease names. No diagnosis. No medical advice. No causes.
- Do not change the priority level.
- 3 to 5 short sentences. Calm and clear.
- If information is missing, say: "Please wait while we complete the next step."
- Output the patient message only.
"""

FEW_SHOT = [
    {"role": "user",  "parts": ["Priority: HIGH\nComplaint: chest pain\nLocation: Emergency, Desk 3\nLanguage: english"]},
    {"role": "model", "parts": ["Your symptoms need prompt medical attention. Please proceed to Emergency, Desk 3 right away. A healthcare professional will see you shortly. If you feel worse at any time, please notify a staff member immediately."]},
    {"role": "user",  "parts": ["Priority: MEDIUM\nComplaint: headache\nLocation: Waiting Area B\nLanguage: english"]},
    {"role": "model", "parts": ["Thank you for sharing your information. Please take a seat in Waiting Area B and a healthcare professional will see you soon. If your condition changes while you wait, please let a staff member know."]},
    {"role": "user",  "parts": ["Priority: LOW\nComplaint: sore throat\nLocation: General Outpatient\nLanguage: english"]},
    {"role": "model", "parts": ["Thank you. Your information has been recorded. Please wait in the General Outpatient area and you will be called when it is your turn. If you have any concerns, please speak to a staff member."]},
]


def build_prompt(inp: GuidanceInput) -> str:
    red_flag = "yes" if inp.red_flags_present else "no" if inp.red_flags_present is False else "unknown"
    return "\n".join([
        f"Priority: {inp.priority}",
        f"Complaint: {inp.chief_complaint or 'not specified'}",
        f"Duration: {inp.duration or 'not specified'}",
        f"Severity: {inp.severity or 'not specified'}",
        f"Red flag: {red_flag}",
        f"Location: {inp.location_context or 'not specified'}",
        f"Language: {inp.patient_language}",
    ])

In [ ]:
def _is_safe(message: str) -> tuple:
    lowered    = message.lower()
    violations = [p for p in FORBIDDEN_PHRASES if p in lowered]
    return len(violations) == 0, violations


def generate_guidance(inp: GuidanceInput) -> dict:
    """
    Generate a patient-facing guidance message.
    Returns dict with 'message', 'safe', 'success'.
    """
    priority  = inp.priority.upper()
    templates = FALLBACK_TEMPLATES_KIN if inp.patient_language == "kinyarwanda" else FALLBACK_TEMPLATES

    if inp.uncertainty_flag or priority not in ("HIGH", "MEDIUM", "LOW"):
        return {"message": templates["UNKNOWN"], "safe": True, "success": True}

    try:
        history = [{"role": "user",  "parts": [SYSTEM_PROMPT]},
                   {"role": "model", "parts": ["Understood. Patient message only, no diagnosis."]}]
        history += FEW_SHOT

        chat     = model.start_chat(history=history)
        response = chat.send_message(build_prompt(inp))
        message  = response.text.strip()
    except Exception as e:
        return {"message": templates.get(priority, templates["UNKNOWN"]), "safe": True, "success": False, "error": str(e)}

    is_safe, violations = _is_safe(message)
    if not is_safe:
        message = templates.get(priority, templates["UNKNOWN"])

    return {"message": message, "safe": is_safe, "violations": violations, "success": True}


def guidance_from_pipeline(model_b_output: dict, model_d_output: dict,
                            patient_language: str = "english", location: str = "") -> dict:
    """Convenience wrapper that accepts Models B and D output dicts directly."""
    ext  = model_b_output.get("extraction_dict", {})
    scr  = model_d_output.get("score_dict", {})
    return generate_guidance(GuidanceInput(
        priority            = scr.get("priority", "MEDIUM"),
        chief_complaint     = ext.get("chief_complaint", ""),
        duration            = ext.get("duration", ""),
        severity            = ext.get("severity", ""),
        associated_symptoms = ext.get("associated_symptoms", []),
        red_flags_present   = ext.get("red_flags_present"),
        patient_language    = patient_language,
        location_context    = location,
        uncertainty_flag    = scr.get("confidence", 1.0) < 0.4,
    ))

---
## Usage

### Option A — Direct input

In [ ]:
result = generate_guidance(GuidanceInput(
    priority="HIGH", chief_complaint="chest pain", duration="2 hours",
    severity="severe", red_flags_present=True,
    patient_language="english", location_context="Emergency, Desk 3",
))
print(result["message"])

### Option B — From pipeline

In [ ]:
# result = guidance_from_pipeline(
#     model_b_output=model_b_result,
#     model_d_output=model_d_result,
#     patient_language="english",
#     location="Waiting Area B",
# )
# print(result["message"])